In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Trỏ thẳng vào thư mục repo đã có sẵn trên Drive
%cd /content/drive/MyDrive
!git clone https://github.com/fyb174147/forecast-models-evaluation.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/forecast-models-evaluation

# Môi trường chạy
- Chạy trên google colab 2026.07 (python 3.13)
- Các model không tương thích tạo 1 environment thông qua condacolab
- Cấu hình: 50GB RAM | GPU T4 (16GB) / L4 (24GB)
    - GPU L4 có tốc độ nhanh hơn 1.5-2x T4 tùy trường hợp
- Lưu ý: colab sẽ tự ngắt kết nối môi trường chạy nếu không sử dụng GPU trong 1-2h hoặc môi trường chạy đã online quá 24h. Trạng thái của môi trường thời gian chạy này, bao gồm tất cả các biến và tệp cục bộ, sẽ bị mất. Nên chia ra chạy từng đợt

# Khảo sát dữ liệu

In [ ]:
import os
import pandas as pd

DATA_DIR = "data"

datasets_to_inspect = {
    # 1. BELGIUM DATASET - Đợt 1 (2024-01-01 đến 2024-04-01: 3 tháng đầu năm)
    "Belgium - House 1 Solar (Jan-Apr 2024)": f"{DATA_DIR}/belgium_dataset/house_1/2024-01-01_0000_2024-04-01_0000/solar.csv",
    "Belgium - House 1 Battery (Jan-Apr 2024)": f"{DATA_DIR}/belgium_dataset/house_1/2024-01-01_0000_2024-04-01_0000/battery.csv",
    "Belgium - House 2 Solar (Jan-Apr 2024)": f"{DATA_DIR}/belgium_dataset/house_2/2024-01-01_0000_2024-04-01_0000/solar.csv",
    "Belgium - House 2 Battery (Jan-Apr 2024)": f"{DATA_DIR}/belgium_dataset/house_2/2024-01-01_0000_2024-04-01_0000/battery.csv",
    "Belgium - House 3 Solar (Jan-Apr 2024)": f"{DATA_DIR}/belgium_dataset/house_3/2024-01-01_0000_2024-04-01_0000/solar.csv",
    "Belgium - House 3 Battery (Jan-Apr 2024)": f"{DATA_DIR}/belgium_dataset/house_3/2024-01-01_0000_2024-04-01_0000/battery.csv",
    "Belgium - House 4 Solar (Jan-Apr 2024)": f"{DATA_DIR}/belgium_dataset/house_4/2024-01-01_0000_2024-04-01_0000/solar.csv",
    "Belgium - House 4 Battery (Jan-Apr 2024)": f"{DATA_DIR}/belgium_dataset/house_4/2024-01-01_0000_2024-04-01_0000/battery.csv",

    # 2. BELGIUM DATASET - Đợt 2 (2024-04-08 đến 2024-06-17: giai đoạn hè)
    "Belgium - House 1 Solar (Apr-Jun 2024)": f"{DATA_DIR}/belgium_dataset/house_1/2024-04-08_1500_2024-06-17_1500/solar.csv",
    "Belgium - House 1 Battery (Apr-Jun 2024)": f"{DATA_DIR}/belgium_dataset/house_1/2024-04-08_1500_2024-06-17_1500/battery.csv",
    "Belgium - House 2 Solar (Apr-Jun 2024)": f"{DATA_DIR}/belgium_dataset/house_2/2024-04-08_1500_2024-06-17_1500/solar.csv",
    "Belgium - House 2 Battery (Apr-Jun 2024)": f"{DATA_DIR}/belgium_dataset/house_2/2024-04-08_1500_2024-06-17_1500/battery.csv",
    "Belgium - House 3 Solar (Apr-Jun 2024)": f"{DATA_DIR}/belgium_dataset/house_3/2024-04-08_1500_2024-06-17_1500/solar.csv",
    "Belgium - House 3 Battery (Apr-Jun 2024)": f"{DATA_DIR}/belgium_dataset/house_3/2024-04-08_1500_2024-06-17_1500/battery.csv",
    "Belgium - House 4 Solar (Apr-Jun 2024)": f"{DATA_DIR}/belgium_dataset/house_4/2024-04-08_1500_2024-06-17_1500/solar.csv",
    "Belgium - House 4 Battery (Apr-Jun 2024)": f"{DATA_DIR}/belgium_dataset/house_4/2024-04-08_1500_2024-06-17_1500/battery.csv",

    # 3. GERMANY WPuQ DATASET (SFH 19 Load)
    "Germany - SFH 19 (3-month 2023-2024 benchmark)": f"{DATA_DIR}/germany_wpuq_dataset/SFH19_2023_2024_15min_3_month.csv",
    "Germany - SFH 19 (Original 2018-2019 1-year)": f"{DATA_DIR}/germany_wpuq_dataset/SFH19_2018_2019_15min_original.csv",

    # 4. LONDON LCL DATASET (LCL 33 Household Load)
    "London - LCL 33 (Cleaned Benchmark 2013)": f"{DATA_DIR}/london_dataset/LCL_london_consumption_2013.csv",
    "London - LCL 33 (Original Raw 2013)": f"{DATA_DIR}/london_dataset/LCL_london_consumption_2013_original.csv",

    # 5. ZONNEDAEL DATASET (Hà Lan - Khách hàng 8, 9, 43...)
    "Zonnedael - Household Customers (Liander 2013)": f"{DATA_DIR}/zonnedael_dataset/liander_zonnedael_2013_original.csv",
}

for name, path in datasets_to_inspect.items():
    if not os.path.exists(path):
        continue
    
    df = pd.read_csv(path)
    
    print("\n" + "="*80)
    print(f"{name.upper()} (File: {os.path.basename(path)})")
    print("="*80)
    
    # df.info()
    print("\n--- [df.info()] ---")
    df.info()
    
    # df.head()
    print("\n--- [df.head()] ---")
    display(df.head(3))

# Tiền xử lí dữ liệu
- Phần tiền xử lí dữ liệu đã được tác giả xử lí tại `utils/dataset_config.py`
- Mỗi khi chạy 1 model việc đầu tiên nó sẽ tự gọi việc tiền xử lí dữ liệu này

# 1. `model_naivedrift.py`

- Model nhẹ, có thể chạy trên CPU

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib
!python models/model_naivedrift.py

# 2. `models_statsml.py`
- Model nhẹ, có thể chạy trên CPU

In [ ]:
!pip install -q statsmodels pmdarima lightgbm xgboost scikit-learn
!python models/models_statsml.py

# 3. `models_neuralforecast.py`
- Bắt buộc dùng GPU có support CUDA core
- Chạy rất lâu và cần nhiều tài nguyên GPU
- Tổng thời gian chạy: 29h với GPU L4 (nhanh hơn khoảng 1.5-2x so với GPU T4)

In [ ]:
!pip install -q neuralforecast pytorch-lightning

# cờ --gpu = [0, -1] tương ứng dùng [GPU, CPU]
# data 1-bỉ, 2-đức, 3-london, 4-zonnedael
# sampling_rate 25, 33.3, 50, 100

!python models/models_neuralforecast.py --gpu 0 --data 1 --sampling_rate 25
!python models/models_neuralforecast.py --gpu 0 --data 1 --sampling_rate 33.333333
!python models/models_neuralforecast.py --gpu 0 --data 1 --sampling_rate 50
!python models/models_neuralforecast.py --gpu 0 --data 1 --sampling_rate 100

!python models/models_neuralforecast.py --gpu 0 --data 2 --sampling_rate 25
!python models/models_neuralforecast.py --gpu 0 --data 2 --sampling_rate 33.333333
!python models/models_neuralforecast.py --gpu 0 --data 2 --sampling_rate 50
!python models/models_neuralforecast.py --gpu 0 --data 2 --sampling_rate 100

!python models/models_neuralforecast.py --gpu 0 --data 3 --sampling_rate 25
!python models/models_neuralforecast.py --gpu 0 --data 3 --sampling_rate 33.333333
!python models/models_neuralforecast.py --gpu 0 --data 3 --sampling_rate 50
!python models/models_neuralforecast.py --gpu 0 --data 3 --sampling_rate 100

!python models/models_neuralforecast.py --gpu 0 --data 4 --sampling_rate 25
!python models/models_neuralforecast.py --gpu 0 --data 4 --sampling_rate 33.333333
!python models/models_neuralforecast.py --gpu 0 --data 4 --sampling_rate 50
!python models/models_neuralforecast.py --gpu 0 --data 4 --sampling_rate 100

# 4. `models_gluonts.py`
- Không tương thích với python 3.13, yêu cầu python 3.7. tạo 1 môi trường python 3.7 qua condacolab
- Có thể chạy với CPU, nhưng mất nhiều thời gian chạy kể cả khi chạy với GPU
- Tổng thời gian chạy: 35h với GPU T4 (không tương thích với GPU L4)

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
!conda create -n gluonts_gpu python=3.7 cudatoolkit=11.2 cudnn=8.1.0 -c conda-forge -y
!source activate gluonts_gpu && pip install mxnet-cu112==1.9.1 gluonts==0.14.4 pandas==1.3.5 numpy==1.21.6 ujson matplotlib scikit-learn
!source activate gluonts_gpu && pip install seaborn==0.12.2 scipy==1.7.3
!source activate gluonts_gpu && pip install matplotlib-inline

In [ ]:
# Sửa lỗi force=True không tương thích với Python 3.7
gluonts_file = "models/models_gluonts.py"

with open(gluonts_file, "r", encoding="utf-8") as f:
    content = f.read()

# Xóa bỏ dòng force=True
new_content = content.replace("        force=True\n", "").replace("        force=True,\n", "")

with open(gluonts_file, "w", encoding="utf-8") as f:
    f.write(new_content)

In [ ]:
import subprocess
import sys

cmd = """
source activate gluonts_gpu
export LD_LIBRARY_PATH=/usr/local/envs/gluonts_gpu/lib:$LD_LIBRARY_PATH
export MPLBACKEND=Agg

python -u models/models_gluonts.py --gpu 0 --data 1 --sampling_rate 25
python -u models/models_gluonts.py --gpu 0 --data 1 --sampling_rate 33.333333
python -u models/models_gluonts.py --gpu 0 --data 1 --sampling_rate 50
python -u models/models_gluonts.py --gpu 0 --data 1 --sampling_rate 100

python -u models/models_gluonts.py --gpu 0 --data 2 --sampling_rate 25
python -u models/models_gluonts.py --gpu 0 --data 2 --sampling_rate 33.333333
python -u models/models_gluonts.py --gpu 0 --data 2 --sampling_rate 50
python -u models/models_gluonts.py --gpu 0 --data 2 --sampling_rate 100

python -u models/models_gluonts.py --gpu 0 --data 3 --sampling_rate 25
python -u models/models_gluonts.py --gpu 0 --data 3 --sampling_rate 33.333333
python -u models/models_gluonts.py --gpu 0 --data 3 --sampling_rate 50
python -u models/models_gluonts.py --gpu 0 --data 3 --sampling_rate 100

python -u models/models_gluonts.py --gpu 0 --data 4 --sampling_rate 25
python -u models/models_gluonts.py --gpu 0 --data 4 --sampling_rate 33.333333
python -u models/models_gluonts.py --gpu 0 --data 4 --sampling_rate 50
python -u models/models_gluonts.py --gpu 0 --data 4 --sampling_rate 100
"""

# Online stream như chạy colab thông thường do condacolab không support online stream

process = subprocess.Popen(
    ["bash", "-c", cmd],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in iter(process.stdout.readline, ''):
    sys.stdout.write(line)
    sys.stdout.flush()

process.stdout.close()
process.wait()

# 5. `model_mamba.py`
- Không tương thích với python 3.13, hạ cấp về python 3.11
- Có thể hạ cấp phiên bản colab 2025.07 (python 3.11) và chạy trực tiếp
- Tổng thời gian: 8h với GPU L4

In [ ]:
!pip install -q torch==2.3.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install --no-deps https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.4.0/causal_conv1d-1.4.0+cu122torch2.3cxx11abiFALSE-cp311-cp311-linux_x86_64.whl
!pip install --no-deps https://github.com/state-spaces/mamba/releases/download/v2.2.2/mamba_ssm-2.2.2+cu122torch2.3cxx11abiFALSE-cp311-cp311-linux_x86_64.whl

In [ ]:
!python models/model_mamba.py --gpu 0 --data 1 --sampling_rate 25
!python models/model_mamba.py --gpu 0 --data 1 --sampling_rate 33.333333
!python models/model_mamba.py --gpu 0 --data 1 --sampling_rate 50
!python models/model_mamba.py --gpu 0 --data 1 --sampling_rate 100

!python models/model_mamba.py --gpu 0 --data 2 --sampling_rate 25
!python models/model_mamba.py --gpu 0 --data 2 --sampling_rate 33.333333
!python models/model_mamba.py --gpu 0 --data 2 --sampling_rate 50
!python models/model_mamba.py --gpu 0 --data 2 --sampling_rate 100

!python models/model_mamba.py --gpu 0 --data 3 --sampling_rate 25
!python models/model_mamba.py --gpu 0 --data 3 --sampling_rate 33.333333
!python models/model_mamba.py --gpu 0 --data 3 --sampling_rate 50
!python models/model_mamba.py --gpu 0 --data 3 --sampling_rate 100

!python models/model_mamba.py --gpu 0 --data 4 --sampling_rate 25
!python models/model_mamba.py --gpu 0 --data 4 --sampling_rate 33.333333
!python models/model_mamba.py --gpu 0 --data 4 --sampling_rate 50
!python models/model_mamba.py --gpu 0 --data 4 --sampling_rate 100

# 6. `fmodel_chronos.py`

In [ ]:
!pip install -q chronos-forecasting

In [ ]:
!python models/fmodel_chronos.py --gpu 0 --data 1 --sampling_rates 25
!python models/fmodel_chronos.py --gpu 0 --data 1 --sampling_rates 33.333333
!python models/fmodel_chronos.py --gpu 0 --data 1 --sampling_rates 50
!python models/fmodel_chronos.py --gpu 0 --data 1 --sampling_rates 100

!python models/fmodel_chronos.py --gpu 0 --data 2 --sampling_rates 25
!python models/fmodel_chronos.py --gpu 0 --data 2 --sampling_rates 33.333333
!python models/fmodel_chronos.py --gpu 0 --data 2 --sampling_rates 50
!python models/fmodel_chronos.py --gpu 0 --data 2 --sampling_rates 100

!python models/fmodel_chronos.py --gpu 0 --data 3 --sampling_rates 25
!python models/fmodel_chronos.py --gpu 0 --data 3 --sampling_rates 33.333333
!python models/fmodel_chronos.py --gpu 0 --data 3 --sampling_rates 50
!python models/fmodel_chronos.py --gpu 0 --data 3 --sampling_rates 100

!python models/fmodel_chronos.py --gpu 0 --data 4 --sampling_rates 25
!python models/fmodel_chronos.py --gpu 0 --data 4 --sampling_rates 33.333333
!python models/fmodel_chronos.py --gpu 0 --data 4 --sampling_rates 50
!python models/fmodel_chronos.py --gpu 0 --data 4 --sampling_rates 100

# 7. `fmodel_timesfm.py`
- Không tương thích python 3.13, tạo môi trường python 3.11 qua condacolab

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
!conda create -y -n timesfm_env python=3.11

!source activate timesfm_env && pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!source activate timesfm_env && pip install -q "timesfm[torch]==1.3.0"
!source activate timesfm_env && pip install matplotlib seaborn

In [ ]:
import subprocess
import sys

cmd = """
source activate timesfm_env
export MPLBACKEND=Agg

python -u models/fmodel_timesfm.py --gpu 0
"""

process = subprocess.Popen(
    ["bash", "-c", cmd],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in iter(process.stdout.readline, ''):
    sys.stdout.write(line)
    sys.stdout.flush()

process.stdout.close()
process.wait()

# 8. `fmodel_moirai.py`
- Không tương thích với python 3.13, tạo môi trường python 3.11 qua condacolab

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
%%bash
conda create -y -n moirai_env python=3.11
source activate moirai_env

pip install "torch==2.4.1+cu121" "torchvision==0.19.1+cu121" "torchaudio==2.4.1+cu121" --index-url https://download.pytorch.org/whl/cu121
pip install "numpy==1.26.4" "scipy==1.11.4" "fsspec<=2023.10.0" "pandas<2.2.0"
pip install --no-deps "gluonts>=0.14.3,<0.15" "uni2ts==2.0.0"
pip install \
    "jaxtyping==0.2.28" \
    "jax==0.4.35" \
    "jaxlib==0.4.35" \
    "datasets~=2.17.1" \
    "einops==0.7.0" \
    "hydra-core==1.3.2" \
    "python-dotenv==1.0.0" \
    "toolz<1.0.0,>=0.10.0" \
    lightning torchmetrics safetensors tensorboard multiprocess orjson ujson \
    pydantic tqdm typing-extensions matplotlib seaborn scikit-learn huggingface-hub

pip install "pandas==2.1.4" "numpy==1.26.4"

In [ ]:
import subprocess
import sys

cmd = """
source activate moirai_env
export MPLBACKEND=Agg

python -u models/fmodel_moirai.py --gpu 0 --data 1
python -u models/fmodel_moirai.py --gpu 0 --data 2
python -u models/fmodel_moirai.py --gpu 0 --data 3
python -u models/fmodel_moirai.py --gpu 0 --data 4
"""

process = subprocess.Popen(
    ["bash", "-c", cmd],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in iter(process.stdout.readline, ''):
    sys.stdout.write(line)
    sys.stdout.flush()

process.stdout.close()
process.wait()

# 9. `fmodel_timerxl.py`
- Không tương thích với python 3.13, tạo môi trường python 3.11 qua condacolab

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
!conda create -y -n timerxl_env python=3.11

!source activate timerxl_env && conda install -y -c pytorch -c nvidia pytorch torchvision pytorch-cuda=12.1
!source activate timerxl_env && pip install "transformers==4.40.1" accelerate safetensors huggingface-hub pandas numpy matplotlib seaborn scikit-learn
!source activate timerxl_env && pip install --upgrade --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
import subprocess
import sys

cmd = """
source activate timerxl_env
export MPLBACKEND=Agg

python -u models/fmodel_timerxl.py --gpu 0 --data 1
python -u models/fmodel_timerxl.py --gpu 0 --data 2
python -u models/fmodel_timerxl.py --gpu 0 --data 3
python -u models/fmodel_timerxl.py --gpu 0 --data 4
"""

process = subprocess.Popen(
    ["bash", "-c", cmd],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in iter(process.stdout.readline, ''):
    sys.stdout.write(line)
    sys.stdout.flush()

process.stdout.close()
process.wait()

# 10. `fmodel_timegpt.py`
- Yêu cầu timegpt api key, lấy tại https://www.nixtla.io/dashboard/api-keys
- Cần thêm api key vào trong colab

In [ ]:
!pip install -q nixtla python-dotenv

In [ ]:
import os
from google.colab import userdata

os.environ["TIMEGPT_API_KEY"] = userdata.get("TIMEGPT_API_KEY")

!python models/fmodel_timegpt.py --dataset belgium
!python models/fmodel_timegpt.py --dataset germany
!python models/fmodel_timegpt.py --dataset london
!python models/fmodel_timegpt.py --dataset zonnedael